In [2]:
import pandas as pd
import os
import pinecone
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

load_dotenv()

True

In [3]:
files = pd.read_csv("course_section_descriptions.csv", encoding="cp1252")

In [4]:
files["unique_id"] = (
    files["course_id"].astype(str) + "-" + files["section_id"].astype(str)
)

In [5]:
files["metadata"] = files.apply(
    lambda row: {
        "course_name": row["course_name"],
        "section_name": row["section_name"],
        "section_description": row["section_description"],
    },
    axis=1,
)

In [6]:
files.head()

,course_id,course_name,course_slug,course_description,course_description_short,course_technology,course_topic,course_instructor_quote,section_id,section_name,section_description,unique_id,metadata
0,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,9,Introduction to Tableau,While Tableau is an indispensable tool in the ...,2-9,"{'course_name': 'Introduction to Tableau', 'se..."
1,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,10,Tableau Functionalities,"In this section, you will create your first Ta...",2-10,"{'course_name': 'Introduction to Tableau', 'se..."
2,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,11,The Tableau Exercise,This section is a practical example that will ...,2-11,"{'course_name': 'Introduction to Tableau', 'se..."
3,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,12,Introduction,"In this section, you will learn about the impo...",3-12,{'course_name': 'The Complete Data Visualizati...
4,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,13,Setting Up the Environments,"Here, we set up different environments for the...",3-13,{'course_name': 'The Complete Data Visualizati...


In [7]:
model = SentenceTransformer("multi-qa-distilbert-cos-v1")

In [ ]:
weight_course_name = 5
weight_section_name = 3
weight_section_description = 2
weight_other = 1

In [9]:
def create_embeddings(row):
    course_name_embedding = (
        model.encode(row["course_name"], show_progress_bar=False) * weight_course_name
    )
    course_description_embedding = (
        model.encode(row["course_description"], show_progress_bar=False) * weight_other
    )
    section_name_embedding = (
        model.encode(row["section_name"], show_progress_bar=False) * weight_section_name
    )
    section_description_embedding = (
        model.encode(row["section_description"], show_progress_bar=False)
        * weight_section_description
    )
    technology_embedding = (
        model.encode(row["course_technology"], show_progress_bar=False) * weight_other
    )

    combined_embedding = (
        course_name_embedding
        + course_description_embedding
        + section_name_embedding
        + section_description_embedding
        + technology_embedding
    )

    avg_embedding = combined_embedding / (
        weight_course_name
        + (2 * weight_other)
        + weight_section_name
        + weight_section_description
    )

    return avg_embedding.tolist()

In [10]:
files["embeddings"] = files.apply(create_embeddings, axis=1)
files.head()

,course_id,course_name,course_slug,course_description,course_description_short,course_technology,course_topic,course_instructor_quote,section_id,section_name,section_description,unique_id,metadata,embeddings
0,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,9,Introduction to Tableau,While Tableau is an indispensable tool in the ...,2-9,"{'course_name': 'Introduction to Tableau', 'se...","[0.007547455374151468, 0.05743563547730446, 0...."
1,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,10,Tableau Functionalities,"In this section, you will create your first Ta...",2-10,"{'course_name': 'Introduction to Tableau', 'se...","[0.009785618633031845, 0.05833655595779419, 0...."
2,2,Introduction to Tableau,tableau,Tableau is now one of the most popular busines...,Teaching you how to tell compelling stories wi...,tableau,data visualization,Data scientists don’t just need to deal with d...,11,The Tableau Exercise,This section is a practical example that will ...,2-11,"{'course_name': 'Introduction to Tableau', 'se...","[0.005808187648653984, 0.05435327813029289, 0...."
3,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,12,Introduction,"In this section, you will learn about the impo...",3-12,{'course_name': 'The Complete Data Visualizati...,"[0.0086356271058321, 0.042541418224573135, 0.0..."
4,3,The Complete Data Visualization Course with Py...,data-visualization,The Data Visualization course is designed for ...,Teaching you how to master the art of creating...,python,data visualization,Data visualization is the face of data. Many p...,13,Setting Up the Environments,"Here, we set up different environments for the...",3-13,{'course_name': 'The Complete Data Visualizati...,"[0.027643069624900818, 0.05211261287331581, 0...."


In [11]:
pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY"), environment=os.getenv("PINECONE_ENVIRONMENT")
)

In [12]:
index_name = "bert-weighted"
dimensions = model.get_sentence_embedding_dimension()
metric = "cosine"

In [13]:
if index_name in [index.name for index in pc.list_indexes()]:
    print(f"Index '{index_name}' already exists.")
    pc.delete_index(index_name)
    print(f"Index '{index_name}' deleted.")
else:
    print(f"Index '{index_name}' does not exist.")

Index 'bert-weighted' does not exist.


In [15]:
pc.create_index(
    name=index_name,
    dimension=dimensions,
    metric=metric,
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

{
    "name": "bert-weighted",
    "metric": "cosine",
    "host": "bert-weighted-h2xw5gc.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "access-control-allow-origin": "*",
            "vary": "origin,access-control-request-method,access-control-request-headers",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "20

In [16]:
index = pc.Index(index_name)

In [17]:
vectors_to_upsert = [
    (row["unique_id"], row["embeddings"], row["metadata"])
    for _, row in files.iterrows()
]

In [18]:
index.upsert(vectors=vectors_to_upsert)
print("Vectors upserted successfully.")

Vectors upserted successfully.


In [19]:
query = "regression"
query_embedding = model.encode(query).tolist()

In [20]:
query_results = index.query(
    vector=[query_embedding], top_k=12, include_metadata=True, include_values=True
)

In [21]:
for match in query_results.matches:
    course_details = match.get("metadata", {})
    course_name = course_details.get("course_name", "N/A")
    section_name = course_details.get("section_name", "N/A")
    section_description = course_details.get("section_description", "N/A")

    print(f"Match ID: {match['id']}, Score: {match['score']:.4f}")
    print(f"Course Name: {course_name}")
    print(f"Section Name: {section_name}")
    print(f"Section Description: {section_description}")
    print()

Match ID: 51-465, Score: 0.5872
Course Name: Machine Learning in Excel
Section Name: Simple Linear Regression
Section Description: Join us to create your first simple regression in Excel and get familiar with a very important statistical concept – the Ordinary least squares framework. You will learn about OLS assumptions, how to interpret regression results, as well as how to decompose variability. 

Match ID: 101-704, Score: 0.5649
Course Name: The Machine Learning Algorithms A-Z
Section Name: Linear Regression
Section Description: Linear regression is the most dynamic model out of all we review. It’s an exceptional framework for making predictions and extracting insight into relationships between variables.

Match ID: 51-468, Score: 0.4928
Course Name: Machine Learning in Excel
Section Name: Logistic Regression
Section Description: This section of the course covers logistic regression. You will grasp the difference between logistic and logit regression, the concepts of ROC curve, und